# Week 3 — SQL Queries (Part A)
NextGenLearners program data: `applicants` → `interns` (via `applicant_id`) → `hackathon_scores` (via `intern_id`).

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('nextgen.db')

def q(sql):
    return pd.read_sql(sql, conn)

## Step 2 — Explore each table first

In [ ]:
q('SELECT * FROM applicants LIMIT 5;')

,applicant_id,name,domain,university,application_date,status
0,1,Usman Shaikh,Data Science,Mehran University,2026-01-09,Selected
1,2,Bilal Shaikh,Data Science,Shah Abdul Latif University,2026-02-05,Selected
2,3,Tariq Ansari,Cybersecurity,Shah Abdul Latif University,2026-02-18,Selected
3,4,Hassan Shaikh,Cybersecurity,COMSATS,2026-01-07,Selected
4,5,Mehwish Channa,Data Science,IBA Karachi,2026-01-06,Rejected


In [ ]:
q('SELECT * FROM interns LIMIT 5;')

,intern_id,applicant_id,domain,start_date,completion_status
0,1,1,Data Science,2026-03-11,Completed
1,2,2,Data Science,2026-03-11,Completed
2,3,3,Cybersecurity,2026-03-12,Dropped Out
3,4,4,Cybersecurity,2026-03-02,Completed
4,5,6,UI/UX Design,2026-03-11,Completed


In [ ]:
q('SELECT * FROM hackathon_scores LIMIT 5;')

,intern_id,score,domain
0,1,67.2,Data Science
1,2,76.2,Data Science
2,4,67.6,Cybersecurity
3,5,88.5,UI/UX Design
4,6,78.3,AI/ML


**Relationships:** `applicants.applicant_id` → `interns.applicant_id`; `interns.intern_id` → `hackathon_scores.intern_id`.

In [ ]:
# Sanity check: row counts per table
q('''
SELECT 'applicants' AS tbl, COUNT(*) AS n FROM applicants
UNION ALL SELECT 'interns', COUNT(*) FROM interns
UNION ALL SELECT 'hackathon_scores', COUNT(*) FROM hackathon_scores;
''')

,tbl,n
0,applicants,300
1,interns,142
2,hackathon_scores,102


## Query 1 — How many interns completed each domain's program?

In [ ]:
q('''
-- Counts completed interns grouped by domain
SELECT domain, COUNT(*) AS completed_count
FROM interns
WHERE completion_status = 'Completed'
GROUP BY domain
ORDER BY completed_count DESC;
''')

,domain,completed_count
0,UI/UX Design,27
1,Web Development,25
2,Data Science,24
3,Cybersecurity,21
4,AI/ML,15


## Query 2 — Average hackathon score per domain

In [ ]:
q('''
-- Calculates the average hackathon score for each domain
SELECT domain, ROUND(AVG(score), 2) AS avg_score
FROM hackathon_scores
GROUP BY domain
ORDER BY avg_score DESC;
''')

,domain,avg_score
0,UI/UX Design,84.05
1,Data Science,77.15
2,AI/ML,74.06
3,Web Development,73.13
4,Cybersecurity,67.60


## Query 3 — Top performers (JOIN)

In [ ]:
q('''
-- Lists interns who scored 85 or above, as candidates for showcase/certificates
SELECT i.intern_id, i.domain, h.score
FROM interns i
JOIN hackathon_scores h ON i.intern_id = h.intern_id
WHERE h.score >= 85
ORDER BY h.score DESC;
''')

,intern_id,domain,score
0,10,Web Development,100.0
1,107,Data Science,96.0
2,131,UI/UX Design,95.0
3,112,UI/UX Design,94.7
4,62,Data Science,94.6
5,116,Data Science,94.1
6,135,UI/UX Design,92.1
7,22,UI/UX Design,91.9
8,133,AI/ML,91.9
9,13,UI/UX Design,90.8


## Query 4 — Conversion rate from applied → completed per domain (LEFT JOIN)

In [ ]:
q('''
-- Compares total applicants vs completed interns per domain to find conversion rate
SELECT
    a.domain,
    COUNT(DISTINCT a.applicant_id) AS total_applicants,
    COUNT(DISTINCT i.intern_id)    AS total_completed,
    ROUND(100.0 * COUNT(DISTINCT i.intern_id) / COUNT(DISTINCT a.applicant_id), 2) AS conversion_rate_pct
FROM applicants a
LEFT JOIN interns i
    ON a.applicant_id = i.applicant_id AND i.completion_status = 'Completed'
GROUP BY a.domain
ORDER BY conversion_rate_pct DESC;
''')

,domain,total_applicants,total_completed,conversion_rate_pct
0,UI/UX Design,56,27,48.21
1,Web Development,60,25,41.67
2,Data Science,61,24,39.34
3,Cybersecurity,70,21,30.00
4,AI/ML,53,15,28.30


**Why LEFT JOIN?** A regular JOIN would drop every applicant who never became an intern, making the rate misleading. Putting `completion_status = 'Completed'` inside `ON` (not `WHERE`) keeps non-converted applicants in the result.

## Query 5 — My choice: Which university produced the highest-scoring interns? (3-table JOIN)

In [ ]:
q('''
-- Average hackathon score and intern count per university (joins all 3 tables)
SELECT a.university,
       COUNT(*)              AS scored_interns,
       ROUND(AVG(h.score),2) AS avg_score
FROM hackathon_scores h
JOIN interns    i ON h.intern_id    = i.intern_id
JOIN applicants a ON i.applicant_id = a.applicant_id
GROUP BY a.university
ORDER BY avg_score DESC;
''')

,university,scored_interns,avg_score
0,IBA Karachi,6,78.73
1,University of Karachi,13,78.02
2,Shah Abdul Latif University,10,76.14
3,Mehran University,17,75.81
4,NED University,15,75.51
5,Quaid-i-Azam University,9,74.46
6,LUMS,19,74.23
7,COMSATS,13,73.88


## Bonus A — Completed vs dropped out per domain

In [ ]:
q('''
-- Shows completed vs dropped-out interns and the dropout rate per domain
SELECT domain,
       SUM(completion_status = 'Completed')   AS completed,
       SUM(completion_status = 'Dropped Out') AS dropped_out,
       ROUND(100.0 * SUM(completion_status = 'Dropped Out') / COUNT(*), 2) AS dropout_rate_pct
FROM interns
GROUP BY domain
ORDER BY dropout_rate_pct DESC;
''')

,domain,completed,dropped_out,dropout_rate_pct
0,Cybersecurity,21,13,38.24
1,AI/ML,15,5,25.00
2,Web Development,25,7,21.88
3,UI/UX Design,27,3,10.00
4,Data Science,24,2,7.69


## Bonus B — Score distribution

In [ ]:
q('''
-- Counts how many interns fall into each 10-point score range
SELECT CASE WHEN score < 60 THEN 'Below 60'
            WHEN score < 70 THEN '60-70'
            WHEN score < 80 THEN '70-80'
            WHEN score < 90 THEN '80-90'
            ELSE '90-100' END AS score_range,
       COUNT(*) AS interns
FROM hackathon_scores
GROUP BY score_range
ORDER BY MIN(score);
''')

,score_range,interns
0,Below 60,5
1,60-70,26
2,70-80,39
3,80-90,19
4,90-100,13


## Bonus C — Top 10 leaderboard (used by the dashboard)

In [ ]:
q('''
-- Top 10 hackathon performers with their names and universities
SELECT a.name, a.university, i.domain, h.score
FROM hackathon_scores h
JOIN interns    i ON h.intern_id    = i.intern_id
JOIN applicants a ON i.applicant_id = a.applicant_id
ORDER BY h.score DESC
LIMIT 10;
''')

,name,university,domain,score
0,Fatima Malik,LUMS,Web Development,100.0
1,Imran Memon,Quaid-i-Azam University,Data Science,96.0
2,Omar Malik,NED University,UI/UX Design,95.0
3,Hassan Abbasi,COMSATS,UI/UX Design,94.7
4,Hira Khan,Shah Abdul Latif University,Data Science,94.6
5,Omar Abbasi,IBA Karachi,Data Science,94.1
6,Bilal Channa,COMSATS,UI/UX Design,92.1
7,Mehwish Ansari,University of Karachi,UI/UX Design,91.9
8,Faisal Shaikh,Mehran University,AI/ML,91.9
9,Zainab Khan,University of Karachi,UI/UX Design,90.8
